In [8]:
from pathlib import Path
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not locate the project root containing src/")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.models import Siamese2D
from src.losses import contrastive_loss
from src.image_dataset import SiameseImagePairDataset

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

IMAGE_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "images"
    / "gadf"
)

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Image directory:", IMAGE_DIR)

Project root: c:\Users\asus\OneDrive\Desktop\siamese-network
Device: cpu
Image directory: c:\Users\asus\OneDrive\Desktop\siamese-network\datasets\images\gadf


In [9]:
model = Siamese2D(
    embedding_dim=128
).to(DEVICE)

In [10]:
classes = ["normal", "irf", "orf"]

NORMALIZED_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "normalized"
)

X_list = []
y_list = []

for label, class_name in enumerate(classes):

    X_class = np.loadtxt(
        NORMALIZED_DIR / f"{class_name}.csv",
        delimiter=","
    ).astype(np.float32)

    X_list.append(X_class)
    y_list.extend(
        [label] * len(X_class)
    )

X = np.concatenate(X_list, axis=0)
y = np.asarray(y_list)

split = np.load(
    PROJECT_ROOT
    / "results"
    / "fsl_split.npz"
)

train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

train_dataset = SiameseImagePairDataset(
    IMAGE_DIR,
    train_idx,
    y,
    classes
)

val_dataset = SiameseImagePairDataset(
    IMAGE_DIR,
    val_idx,
    y,
    classes
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = MODEL_DIR / "fsl_gadf_best.pt"

print("Training pairs:", len(train_dataset))
print("Validation pairs:", len(val_dataset))

Training pairs: 1680
Validation pairs: 360


In [11]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    device
):

    model.train()
    total_loss = 0.0

    for x1, x2, label in loader:

        x1 = x1.to(device)
        x2 = x2.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        z1, z2 = model(x1, x2)
        loss = contrastive_loss(z1, z2, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate(
    model,
    loader,
    device
):

    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for x1, x2, label in loader:

            x1 = x1.to(device)
            x2 = x2.to(device)
            label = label.to(device)

            z1, z2 = model(x1, x2)
            loss = contrastive_loss(z1, z2, label)
            total_loss += loss.item()

    return total_loss / len(loader)


EPOCHS = 20
best_val_loss = float("inf")

if CHECKPOINT_PATH.exists():

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False
    )
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    print(
        "Existing GADF checkpoint loaded; training skipped."
    )

else:

    for epoch in range(1, EPOCHS + 1):

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            DEVICE
        )

        val_loss = validate(
            model,
            val_loader,
            DEVICE
        )

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train loss: {train_loss:.6f} | "
            f"Val loss: {val_loss:.6f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss
                },
                CHECKPOINT_PATH
            )

            print("Checkpoint saved:", CHECKPOINT_PATH)

Epoch 01/20 | Train loss: 0.317596 | Val loss: 0.262137
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gadf_best.pt
Epoch 02/20 | Train loss: 0.257714 | Val loss: 0.272869
Epoch 03/20 | Train loss: 0.252488 | Val loss: 0.263428
Epoch 04/20 | Train loss: 0.245122 | Val loss: 0.280063
Epoch 05/20 | Train loss: 0.237884 | Val loss: 0.261961
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gadf_best.pt
Epoch 06/20 | Train loss: 0.229008 | Val loss: 0.262405
Epoch 07/20 | Train loss: 0.214331 | Val loss: 0.273799
Epoch 08/20 | Train loss: 0.207577 | Val loss: 0.255648
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gadf_best.pt
Epoch 09/20 | Train loss: 0.195738 | Val loss: 0.253634
Checkpoint saved: c:\Users\asus\OneDrive\Desktop\siamese-network\models\fsl_gadf_best.pt
Epoch 10/20 | Train loss: 0.189467 | Val loss: 0.267074
Epoch 11/20 | Train loss: 0.187093 | Val loss: 0.252636
Checkpoint saved: c:\Users\a

In [ ]:
import json

from src.image_fsl import evaluate_image_fsl

model.load_state_dict(
    torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False
    )["model_state_dict"]
)

fsl_results = {
    "representation": "GADF",
    "model": "Siamese Conv2D",
}

for k_shot in (1, 5):

    metrics = evaluate_image_fsl(
        model=model,
        image_root=IMAGE_DIR,
        test_indices=test_idx,
        labels=y,
        class_names=classes,
        device=DEVICE,
        k_shot=k_shot,
        n_episodes=100,
        query_per_class=5,
        seed=42
    )

    fsl_results[f"{k_shot}_shot"] = {
        "accuracy": float(metrics["accuracy"]),
        "f1": float(metrics["f1"]),
        "auc": float(metrics["auc"])
    }

    print(
        f"{k_shot}-shot:",
        fsl_results[f"{k_shot}_shot"]
    )

with (
    PROJECT_ROOT
    / "results"
    / "gadf_fsl_results.json"
).open("w", encoding="utf-8") as file:
    json.dump(fsl_results, file, indent=4)

print("Saved:", PROJECT_ROOT / "results" / "gadf_fsl_results.json")

1-shot: {'accuracy': 0.464, 'f1': 0.4670463801184277, 'auc': 0.6543113333333334}
5-shot: {'accuracy': 0.5186666666666667, 'f1': 0.5191492007426234, 'auc': 0.7272006666666666}
Saved: c:\Users\asus\OneDrive\Desktop\siamese-network\results\gadf_fsl_results.json


: 